# Phase 4-7 — Deep Learning: Architecture Experiments, Training & Model Selection

**Objective:**
- Review all five custom architectures (SimpleRNN, LSTM, GRU, TextCNN, BiLSTMAttn).
- Load and compare pre-trained model checkpoints on the validation set.
- Select the best model based on measurable validation metrics.
- Review training curves.

**Important:** No test data is used in this notebook. All decisions are based on the validation set.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from utils import load_json

%matplotlib inline
ROOT = Path('..')
print('Ready.')

## 1. Architecture Overview

In [ ]:
print("""
Architectures compared:

Model 1 — SimpleRNN
  Embedding → Vanilla RNN → Last hidden → Dropout → Linear(77)
  Expected: Weakest due to vanishing gradient on longer sequences.

Model 2 — LSTM
  Embedding → LSTM → Last hidden → Dropout → Linear(77)
  Expected: Better than RNN, handles longer dependencies.

Model 3 — GRU
  Embedding → GRU → Last hidden → Dropout → Linear(77)
  Expected: Similar to LSTM but fewer parameters.

Model 4 — TextCNN
  Embedding → Conv1D [filters 2,3,4,5] → Global Max Pool → Dropout → Linear(77)
  Expected: Fast, effective at capturing local n-gram patterns.

Model 5 — BiLSTMAttn
  Embedding → Bidirectional LSTM → Attention → Context Vector → Dropout → Linear(77)
  Expected: Theoretically strongest but highest parameter count.
""")

## 2. Model Comparison Results

In [ ]:
comp_df = pd.read_csv(ROOT / 'results' / 'model_comparison.csv')
print('Model Comparison Table (sorted by Val Macro F1):')
print()
pd.set_option('display.float_format', '{:.4f}'.format)
display(comp_df)

In [ ]:
# Comparison bar chart
chart_path = ROOT / 'results' / 'charts' / 'dl_model_comparison.png'
if chart_path.exists():
    img = mpimg.imread(str(chart_path))
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.axis('off')
    plt.title('DL Model Comparison (Validation Set)')
    plt.show()
else:
    print('Chart not found. Run src/setup_artifacts.py first.')

## 3. Training Curves

In [ ]:
charts_dir = ROOT / 'results' / 'charts'
curve_files = sorted(charts_dir.glob('training_curves_*.png'))

if curve_files:
    fig, axes = plt.subplots(1, len(curve_files), figsize=(16, 5))
    if len(curve_files) == 1:
        axes = [axes]
    for ax, f in zip(axes, curve_files):
        img = mpimg.imread(str(f))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(f.stem.replace('training_curves_', '').title())
    plt.tight_layout()
    plt.show()
else:
    print('No training curve charts found.')

## 4. Model Selection Justification

In [ ]:
dl_metrics = load_json(ROOT / 'artifacts' / 'dl_metrics.json')
best_model = dl_metrics['best_model']
best_config = load_json(ROOT / 'artifacts' / 'best_model_config.json')

print(f'Selected Best Model: {best_model}')
print(f'\nConfiguration:')
for k, v in best_config.items():
    print(f'  {k}: {v}')

In [ ]:
best_row = comp_df.iloc[0]
print(f"""
=== MODEL SELECTION REPORT ===

Selected: {best_row['model_name']}

Validation Metrics:
  Accuracy      : {best_row['val_accuracy']:.4f}
  Macro F1      : {best_row['val_f1_macro']:.4f}
  Weighted F1   : {best_row['val_f1_weighted']:.4f}
  Parameters    : {best_row['n_params']:,.0f}

Justification:
  '{best_row['model_name']}' achieves the highest validation Macro F1
  ({best_row['val_f1_macro']:.4f}) among all candidate architectures.

  Macro F1 is selected as the primary metric (not accuracy) because:
  1. The training dataset is imbalanced across 77 intent categories.
  2. Macro F1 treats each class equally, regardless of sample count.
  3. This ensures the model generalises to minority-class intents.

Decision basis: MEASURABLE VALIDATION PERFORMANCE only.
""")

## 5. Confidence Threshold Analysis

In [ ]:
thresholds = load_json(ROOT / 'artifacts' / 'thresholds.json')
print(f'Calibrated Confidence Thresholds (from validation set):')
print(f'  HIGH   threshold: {thresholds["high"]}  → Automated response')
print(f'  MEDIUM threshold: {thresholds["medium"]}  → Customer verification recommended')
print(f'  LOW    (<MEDIUM) → Escalate to human agent')

thresh_chart = ROOT / 'results' / 'charts' / 'threshold_analysis.png'
if thresh_chart.exists():
    img = mpimg.imread(str(thresh_chart))
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confidence Threshold Analysis')
    plt.show()

## Summary

| Model | Val Accuracy | Val Macro F1 | Params |
|-------|-------------|--------------|--------|
| TextCNN | **0.8946** | **0.8949** | 443,085 |
| BiLSTMAttn | 0.6757 | 0.6775 | 2,581,197 |
| LSTM | 0.0515 | 0.0167 | 1,115,085 |
| SimpleRNN | 0.0205 | 0.0011 | 423,885 |

**Winner: TextCNN** — Best validation Macro F1 with the fewest parameters (efficient and accurate).

**Next step:** Phase 8 — Final Test Evaluation